# AI4I 2020 Predictive Maintenance with TSFresh

This notebook demonstrates an ML workflow specifically using `tsfresh` to extract time-series features. We treat the sequence of products (`UDI`) as a single continuous time series generated by the machine (since tool wear increases over time). We will use a rolling window to extract time-series features for out-of-sample forecasting, followed by hyperparameter tuning.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, TimeSeriesSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

import tsfresh
from tsfresh import extract_features
from tsfresh.utilities.dataframe_functions import roll_time_series
from tsfresh.feature_extraction import MinimalFCParameters

## 1. Data Pre-Processing for TSFresh
`tsfresh` requires data in a specific long format with a distinct `id` for each time series and a `time` column. Since this is one continuous machine process, we assign a global `id = 1` and use `UDI` as the `time` step.

In [ ]:
# Load data
data_path = 'data/01_raw/pred-maintenance/ai4i2020.csv'
df = pd.read_csv(data_path)

# Drop leakage and non-predictive columns
target_col = 'Machine failure'
leakage_cols = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']
df = df.drop(columns=leakage_cols + ['Product ID'])

# Encode the ordinal 'Type' feature
df['Type'] = df['Type'].map({'L': 0, 'M': 1, 'H': 2})

# Set up tsfresh variables
df['id'] = 1
df['time'] = df['UDI']

df_ts = df[['id', 'time', 'Type', 'Air temperature [K]', 'Process temperature [K]', 
            'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']]

display(df_ts.head())

,id,time,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min]
0,1,1,1,298.1,308.6,1551,42.8,0
1,1,2,0,298.2,308.7,1408,46.3,3
2,1,3,0,298.1,308.5,1498,49.4,5
3,1,4,0,298.2,308.6,1433,39.5,7
4,1,5,0,298.2,308.7,1408,40.0,9


## 2. Rolling Time Series & Feature Extraction
To predict failure at time `T` using the sequence of past events, we create rolling windows. We will use a `max_timeshift` of 4 (so each window looks at the current step and up to 4 past steps) to avoid exploding the dataset size.

In [3]:
print("Rolling time series...")
# Create rolling windows
df_rolled = roll_time_series(df_ts, column_id='id', column_sort='time', 
                             max_timeshift=4, min_timeshift=0)

print("Extracting features...")
# We use MinimalFCParameters to keep extraction time extremely fast (extracts mean, median, min, max, std, variance, etc.)
# For deeper features, you could use EfficientFCParameters()
extraction_settings = MinimalFCParameters()
X_tsfresh = extract_features(df_rolled, column_id='id', column_sort='time', 
                             default_fc_parameters=extraction_settings, disable_progressbar=False)

# The index of X_tsfresh is a tuple of (id, time) generated by roll_time_series.
# We extract just the 'time' (UDI) part of the tuple to align with our target variable.
X_tsfresh.index = [idx[1] for idx in X_tsfresh.index]
X_tsfresh.index.name = 'time'

display(X_tsfresh.head())

Rolling time series...


Rolling: 100%|██████████| 40/40 [00:16<00:00,  2.42it/s]


Extracting features...


Feature Extraction: 100%|██████████| 40/40 [00:28<00:00,  1.41it/s]


,Type__sum_values,Type__median,Type__mean,Type__length,Type__standard_deviation,Type__variance,Type__root_mean_square,Type__maximum,Type__absolute_maximum,Type__minimum,...,Tool wear [min]__sum_values,Tool wear [min]__median,Tool wear [min]__mean,Tool wear [min]__length,Tool wear [min]__standard_deviation,Tool wear [min]__variance,Tool wear [min]__root_mean_square,Tool wear [min]__maximum,Tool wear [min]__absolute_maximum,Tool wear [min]__minimum
time,,,,,,,,,,,,,,,,,,,,,
1,1.0,1.0,1.000000,1.0,0.000000,0.000000,1.000000,1.0,1.0,1.0,...,0.0,0.0,0.000000,1.0,0.000000,0.000000,0.000000,0.0,0.0,0.0
2,1.0,0.5,0.500000,2.0,0.500000,0.250000,0.707107,1.0,1.0,0.0,...,3.0,1.5,1.500000,2.0,1.500000,2.250000,2.121320,3.0,3.0,0.0
3,1.0,0.0,0.333333,3.0,0.471405,0.222222,0.577350,1.0,1.0,0.0,...,8.0,3.0,2.666667,3.0,2.054805,4.222222,3.366502,5.0,5.0,0.0
4,1.0,0.0,0.250000,4.0,0.433013,0.187500,0.500000,1.0,1.0,0.0,...,15.0,4.0,3.750000,4.0,2.586020,6.687500,4.555217,7.0,7.0,0.0
5,1.0,0.0,0.200000,5.0,0.400000,0.160000,0.447214,1.0,1.0,0.0,...,24.0,5.0,4.800000,5.0,3.124100,9.760000,5.727128,9.0,9.0,0.0


## 3. Train/Test Split (Out of Sample Testing)
We drop any missing values generated by the extraction and perform a sequential split to prevent future data from leaking into the past.

In [4]:
# Align target with extracted features
y = df.set_index('time')[target_col].loc[X_tsfresh.index]

# Impute any potential missing values (tsfresh can sometimes output NaNs for variances of length 1, etc.)
X_tsfresh = X_tsfresh.fillna(0)

# Temporal split (80/20) - Out of sample forecasting
X_train, X_test, y_train, y_test = train_test_split(X_tsfresh, y, test_size=0.2, shuffle=False)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

Training set shape: (8000, 60)
Testing set shape: (2000, 60)


## 4. Hyperparameter Tuning & Model Training
We utilize `GridSearchCV` combined with `TimeSeriesSplit` to safely tune hyperparameters on the sequential training set, and evaluate on the strictly out-of-sample test set.

In [5]:
# TimeSeriesSplit ensures cross-validation does not look into the future during tuning
tscv = TimeSeriesSplit(n_splits=3)

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5]
}

rf = RandomForestClassifier(random_state=42)

print("Starting Hyperparameter Tuning...")
grid_search = GridSearchCV(rf, param_grid, cv=tscv, scoring='roc_auc', n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

print(f"\nBest Parameters: {grid_search.best_params_}")

Starting Hyperparameter Tuning...
Fitting 3 folds for each of 18 candidates, totalling 54 fits

Best Parameters: {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 200}


## 5. Evaluation

In [6]:
best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print("Out of Sample Classification Report:\n")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}")

Out of Sample Classification Report:

              precision    recall  f1-score   support

           0       0.98      1.00      0.99      1961
           1       0.40      0.05      0.09        39

    accuracy                           0.98      2000
   macro avg       0.69      0.52      0.54      2000
weighted avg       0.97      0.98      0.97      2000

ROC-AUC Score: 0.9221
